# 🎥 Entrega 1 — OpenPose sobre o REHAB24-6 no Google Colab

Extrai os **keypoints (BODY_25)** dos vídeos de reabilitação do **REHAB24-6** usando a
**GPU gratuita do Colab** (T4, 16 GB). O OpenPose só gera os JSON; a análise (ângulos,
desvios, relatório, validação contra os rótulos) usa o nosso código do repositório.

**Fluxo:** build do OpenPose → baixar o REHAB24-6 (Zenodo) → rodar OpenPose → analisar → baixar.

> ⚠️ **Aviso honesto:** compilar o OpenPose no Colab é historicamente **chato** (o OpenPose
> 1.7 mira CUDA/cuDNN antigos). Se algum passo quebrar, é questão de ajustar flags.

**Runtime → Alterar tipo de ambiente de execução → GPU** antes de começar.

## 0. Conferir a GPU

In [ ]:
!nvidia-smi

## 1. Build do OpenPose (v1.7.0)
Leva ~10–20 min. Compilamos **com CUDA e sem cuDNN** (`-DUSE_CUDNN=OFF`), o contorno mais estável.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y libatlas-base-dev libprotobuf-dev libleveldb-dev \n    libsnappy-dev libhdf5-serial-dev protobuf-compiler libgflags-dev \n    libgoogle-glog-dev liblmdb-dev opencl-headers ocl-icd-opencl-dev libviennacl-dev > /dev/null


In [ ]:
import os
if not os.path.exists('openpose'):
    !git clone --depth 1 -b v1.7.0 https://github.com/CMU-Perceptual-Computing-Lab/openpose.git


In [ ]:
# Modelo BODY_25. O servidor da CMU está fora do ar; usamos o mirror HuggingFace (testado).
%cd /content/openpose/models
!curl -L -o pose/body_25/pose_iter_584000.caffemodel \n  https://huggingface.co/camenduru/openpose/resolve/f4a22b0e6fa2a4a2b1e2d50bd589e8bb11ebea7c/pose_iter_584000.caffemodel
%cd /content/openpose

In [ ]:
!mkdir -p build
%cd build
!cmake .. -DUSE_CUDNN=OFF -DBUILD_PYTHON=OFF -DDOWNLOAD_BODY_25_MODEL=OFF > /tmp/cmake.log 2>&1 || tail -30 /tmp/cmake.log
!make -j`nproc` > /tmp/make.log 2>&1 || tail -40 /tmp/make.log
%cd /content/openpose
!ls build/examples/openpose/openpose.bin && echo '✅ build OK'

## 2. Baixar o REHAB24-6 (Zenodo)
Só os vídeos (`videos.zip`, 2,7 GB) e os rótulos (`Segmentation.csv`). Aberto, sem login.

In [ ]:
%cd /content
!wget -q -O videos.zip 'https://zenodo.org/records/13305826/files/videos.zip?download=1'
!wget -q -O Segmentation.csv 'https://zenodo.org/records/13305826/files/Segmentation.csv?download=1'
!unzip -q -o videos.zip -d rehab24-6
import glob; print(len(glob.glob('rehab24-6/**/*.mp4', recursive=True)), 'vídeos')

In [ ]:
# Escolha quais vídeos processar (Camera17 = horizontal). Aqui: PM_006 (Ex4) e PM_008 (Ex6).
# Na T4 não precisa subamostrar (é rápido); local usamos --frame-step só pela GPU fraca.
import glob
videos = sorted(
    glob.glob('/content/rehab24-6/Ex4/PM_006-Camera17*.mp4') +
    glob.glob('/content/rehab24-6/Ex6/PM_008-Camera17*.mp4')
)
print(videos)

## 3. Rodar o OpenPose → JSON por frame
A T4 aguenta resolução maior que a MX330 local. `-1x256` mantém a proporção.

In [ ]:
import os, subprocess
BIN = '/content/openpose/build/examples/openpose/openpose.bin'
os.makedirs('/content/json', exist_ok=True)

for v in videos:
    name = os.path.splitext(os.path.basename(v))[0]
    out = f'/content/json/{name}'
    os.makedirs(out, exist_ok=True)
    print('▶', name)
    subprocess.run([BIN, '--video', v, '--write_json', out,
                    '--model_pose', 'BODY_25', '--net_resolution', '-1x256',
                    '--display', '0', '--render_pose', '0'],
                   cwd='/content/openpose', check=True)
    print('   frames:', len(os.listdir(out)))

## 4. Analisar com o nosso código (ângulos + desvios + validação)
Clona o repositório do projeto e roda o pipeline `src.video.cli` sobre cada pasta de JSONs.

In [ ]:
%cd /content
![ -d tech-challenge ] || git clone <URL_DO_SEU_REPO_GIT> tech-challenge
%cd /content/tech-challenge
!pip -q install -r requirements.txt

In [ ]:
import glob, os
for jdir in sorted(glob.glob('/content/json/*')):
    name = os.path.basename(jdir)
    # análise + relatório + validação (--segmentation também traz o exercício no relatório).
    # OBS: OpenPose aqui rodou em frames cheios (30 fps), então --fps 30 e sem --frame-step.
    !python -m src.video.cli --json-dir "{jdir}" --fps 30 --out reports --segmentation /content/Segmentation.csv
    # overlay: --json-dir não aceita --overlay, então geramos à parte com o vídeo original
    vids = glob.glob(f'/content/rehab24-6/**/{name}.mp4', recursive=True)
    if vids:
        !python -m src.video.overlay --video "{vids[0]}" --json-dir "{jdir}" --out "reports/{name}_overlay.mp4" --fps 30
!ls -R reports

## 5. Baixar os resultados

In [ ]:
!zip -qr /content/resultados_openpose.zip /content/json reports
from google.colab import files
files.download('/content/resultados_openpose.zip')